# PHOTO MATCHING SỬ DỤNG SIAMESE NETWORK + CNN VỚI PYTORCH

**Bài toán:** So khớp ảnh (Photo Matching) - Xác định 2 ảnh có giống nhau không

**Kiến trúc:** Siamese Network + CNN (ResNet18) + Contrastive Loss

---

## BƯỚC 1: CÀI ĐẶT THƯ VIỆN
Cài đặt các thư viện cần thiết cho dự án (chạy 1 lần nếu chưa có)

In [ ]:
# Cài đặt thư viện cần thiết
!pip install torch torchvision Pillow matplotlib numpy tqdm

## BƯỚC 2: IMPORT THƯ VIỆN
Import tất cả thư viện cần thiết và tự động nhận diện thiết bị CPU/GPU

In [ ]:
# --- Thư viện PyTorch cốt lõi ---
import torch                                        # Framework deep learning chính
import torch.nn as nn                                # Module xây dựng mạng neural network
import torch.nn.functional as F                      # Các hàm activation, loss, v.v.
import torch.optim as optim                          # Các thuật toán tối ưu (Adam, SGD,...)
from torch.utils.data import Dataset, DataLoader     # Đọc và load dữ liệu

# --- Thư viện xử lý ảnh & mô hình pre-trained ---
import torchvision                                   # Datasets, models, transforms cho Computer Vision
import torchvision.transforms as transforms          # Các phép biến đổi ảnh
import torchvision.models as models                  # Các mô hình pre-trained (ResNet, VGG,...)
from PIL import Image                                # Đọc và xử lý file ảnh

# --- Thư viện hỗ trợ ---
import numpy as np                                   # Tính toán ma trận, số học
import matplotlib.pyplot as plt                      # Vẽ biểu đồ, hiển thị ảnh
import os                                            # Thao tác với file/thư mục
import random                                        # Sinh số ngẫu nhiên
from tqdm import tqdm                                # Thanh tiến trình (progress bar)

print(">>> Import thư viện thành công!")

In [ ]:
# --- Tự động nhận diện thiết bị CPU/GPU ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"{'='*50}")
print(f"Thiết bị đang sử dụng: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"{'='*50}")

In [ ]:
# --- Cấu hình đường dẫn dữ liệu ---
DATA_DIR = "data/input"
SIMILAR_DIR = os.path.join(DATA_DIR, "similar")        # Thư mục chứa cặp ảnh giống nhau
DISSIMILAR_DIR = os.path.join(DATA_DIR, "dissimilar")   # Thư mục chứa cặp ảnh khác nhau
MODEL_DIR = "models"                                     # Thư mục lưu model đã train

# Tạo thư mục nếu chưa tồn tại
for dir_path in [SIMILAR_DIR, DISSIMILAR_DIR, MODEL_DIR]:
    os.makedirs(dir_path, exist_ok=True)
    print(f"[OK] Thư mục sẵn sàng: {dir_path}")

print("\n>>> Bước 1 & 2 hoàn tất!")

---
## BƯỚC 3: CHUẨN BỊ DATASET (SiameseDataset)

Tạo class `SiameseDataset` kế thừa từ `torch.utils.data.Dataset`.

**Cách tổ chức dữ liệu:**
- `data/input/similar/`: Chứa các cặp ảnh giống nhau, đặt tên theo quy ước `pair_XXX_a.jpg` và `pair_XXX_b.jpg`
- `data/input/dissimilar/`: Chứa các cặp ảnh khác nhau, đặt tên tương tự

**Label:**
- `0` = Cặp ảnh giống nhau (similar / positive pair)
- `1` = Cặp ảnh khác nhau (dissimilar / negative pair)

> **Lưu ý:** Label = 0 nghĩa là khoảng cách mong muốn = 0 (giống nhau), Label = 1 nghĩa là cần đẩy ra xa (khác nhau). Đây là quy ước theo công thức Contrastive Loss.

In [ ]:
# --- Định nghĩa các phép biến đổi ảnh (Image Transforms) ---
# Chuẩn hóa ảnh đầu vào để phù hợp với mô hình ResNet18 pre-trained

transform = transforms.Compose([
    transforms.Resize((224, 224)),          # Resize ảnh về kích thước 224x224 (chuẩn ImageNet)
    transforms.ToTensor(),                   # Chuyển ảnh PIL thành Tensor (giá trị 0-1)
    transforms.Normalize(                    # Chuẩn hóa theo mean/std của ImageNet
        mean=[0.485, 0.456, 0.406],          # Giá trị trung bình RGB của ImageNet
        std=[0.229, 0.224, 0.225]            # Độ lệch chuẩn RGB của ImageNet
    )
])

print(">>> Transforms đã được định nghĩa.")
print(transform)

In [ ]:
class SiameseDataset(Dataset):
    """
    Dataset cho Siamese Network.
    Đọc các cặp ảnh từ 2 thư mục: similar và dissimilar.
    
    Quy ước đặt tên file trong mỗi thư mục:
        - pair_001_a.jpg, pair_001_b.jpg  (cặp thứ 1)
        - pair_002_a.jpg, pair_002_b.jpg  (cặp thứ 2)
        - ...
    
    Labels:
        - 0: Cặp ảnh giống nhau (similar)  -> Contrastive Loss sẽ kéo lại gần
        - 1: Cặp ảnh khác nhau (dissimilar) -> Contrastive Loss sẽ đẩy ra xa
    """
    
    def __init__(self, similar_dir, dissimilar_dir, transform=None):
        """
        Args:
            similar_dir (str): Đường dẫn thư mục chứa cặp ảnh giống nhau
            dissimilar_dir (str): Đường dẫn thư mục chứa cặp ảnh khác nhau
            transform: Các phép biến đổi ảnh
        """
        self.transform = transform
        self.pairs = []   # Danh sách các cặp: (đường_dẫn_ảnh_A, đường_dẫn_ảnh_B, label)
        
        # --- Đọc cặp ảnh giống nhau (label = 0) ---
        self._load_pairs(similar_dir, label=0)
        
        # --- Đọc cặp ảnh khác nhau (label = 1) ---
        self._load_pairs(dissimilar_dir, label=1)
        
        print(f"[Dataset] Tổng số cặp ảnh: {len(self.pairs)}")
        print(f"  - Similar (label=0): {sum(1 for _, _, l in self.pairs if l == 0)}")
        print(f"  - Dissimilar (label=1): {sum(1 for _, _, l in self.pairs if l == 1)}")
    
    def _load_pairs(self, folder, label):
        """
        Đọc các cặp ảnh từ một thư mục.
        Tìm các file có hậu tố _a và _b cùng prefix để ghép thành cặp.
        """
        if not os.path.exists(folder):
            print(f"[WARNING] Thư mục không tồn tại: {folder}")
            return
        
        # Lấy danh sách tất cả file ảnh trong thư mục
        image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
        all_files = sorted([
            f for f in os.listdir(folder)
            if f.lower().endswith(image_extensions)
        ])
        
        # Tìm các cặp ảnh dựa trên quy ước đặt tên: prefix_a.ext và prefix_b.ext
        pair_dict = {}  # {prefix: {'a': filepath, 'b': filepath}}
        
        for filename in all_files:
            name_without_ext = os.path.splitext(filename)[0]  # Bỏ đuôi file
            
            # Kiểm tra hậu tố _a hoặc _b
            if name_without_ext.endswith('_a'):
                prefix = name_without_ext[:-2]  # Bỏ '_a'
                if prefix not in pair_dict:
                    pair_dict[prefix] = {}
                pair_dict[prefix]['a'] = os.path.join(folder, filename)
                
            elif name_without_ext.endswith('_b'):
                prefix = name_without_ext[:-2]  # Bỏ '_b'
                if prefix not in pair_dict:
                    pair_dict[prefix] = {}
                pair_dict[prefix]['b'] = os.path.join(folder, filename)
        
        # Ghép các cặp hoàn chỉnh (có cả _a và _b)
        for prefix, paths in pair_dict.items():
            if 'a' in paths and 'b' in paths:
                self.pairs.append((paths['a'], paths['b'], label))
        
        print(f"  [OK] Đọc {len([p for p in pair_dict.values() if 'a' in p and 'b' in p])} cặp từ: {folder}")
    
    def __len__(self):
        """Trả về tổng số cặp ảnh trong dataset."""
        return len(self.pairs)
    
    def __getitem__(self, idx):
        """
        Trả về một cặp ảnh và label tương ứng.
        
        Returns:
            img_a (Tensor): Ảnh A đã qua transform
            img_b (Tensor): Ảnh B đã qua transform
            label (Tensor): 0 = giống nhau, 1 = khác nhau
        """
        path_a, path_b, label = self.pairs[idx]
        
        # Đọc ảnh bằng PIL và chuyển sang RGB
        img_a = Image.open(path_a).convert('RGB')
        img_b = Image.open(path_b).convert('RGB')
        
        # Áp dụng transforms (resize, normalize, ...)
        if self.transform:
            img_a = self.transform(img_a)
            img_b = self.transform(img_b)
        
        # Chuyển label thành tensor float
        label = torch.tensor(label, dtype=torch.float32)
        
        return img_a, img_b, label

print(">>> Class SiameseDataset đã được định nghĩa.")

In [ ]:
# --- Khởi tạo Dataset và DataLoader ---

# Tạo dataset từ thư mục dữ liệu
dataset = SiameseDataset(
    similar_dir=SIMILAR_DIR,
    dissimilar_dir=DISSIMILAR_DIR,
    transform=transform
)

# Chia dataset thành tập train (80%) và test (20%)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

print(f"\nChia dataset:")
print(f"  - Train: {train_size} cặp")
print(f"  - Test:  {test_size} cặp")

# Tạo DataLoader để load dữ liệu theo batch
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Xáo trộn dữ liệu mỗi epoch
    num_workers=0        # Số worker đọc dữ liệu song song
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Số batch train: {len(train_loader)}")
print(f"  - Số batch test:  {len(test_loader)}")
print("\n>>> Bước 3 hoàn tất! Dataset đã sẵn sàng.")

---
## BƯỚC 4: XÂY DỰNG MẠNG CNN (Feature Extractor)

Sử dụng **ResNet18 pre-trained** trên ImageNet làm bộ trích xuất đặc trưng:
- Giữ nguyên các lớp Convolution đã được huấn luyện sẵn
- **Bỏ lớp Fully Connected (FC) cuối cùng** → thay bằng lớp FC mới để tạo embedding vector
- Output: Vector đặc trưng **128 chiều** cho mỗi ảnh

In [ ]:
class CNNFeatureExtractor(nn.Module):
    """
    Mạng CNN trích xuất đặc trưng từ ảnh.
    Sử dụng ResNet18 pre-trained, bỏ lớp FC cuối,
    thay bằng lớp FC mới để tạo embedding vector 128 chiều.
    """
    
    def __init__(self, embedding_dim=128):
        """
        Args:
            embedding_dim (int): Số chiều của vector đặc trưng đầu ra
        """
        super(CNNFeatureExtractor, self).__init__()
        
        # Load mô hình ResNet18 đã được train trên ImageNet
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # Lấy tất cả các lớp của ResNet18 TRỪ lớp FC cuối cùng
        # ResNet18 gốc: conv layers -> avgpool -> fc(512, 1000)
        # Ta giữ:        conv layers -> avgpool
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        # Thêm lớp FC mới để tạo embedding vector
        # Input: 512 (output của ResNet18 sau avgpool)
        # Output: embedding_dim (mặc định 128)
        self.fc = nn.Sequential(
            nn.Linear(512, 256),        # Giảm từ 512 xuống 256
            nn.ReLU(),                   # Activation function
            nn.Linear(256, embedding_dim) # Giảm từ 256 xuống 128
        )
    
    def forward(self, x):
        """
        Forward pass: Ảnh đầu vào -> Vector đặc trưng
        
        Args:
            x (Tensor): Batch ảnh, shape [batch_size, 3, 224, 224]
        Returns:
            Tensor: Embedding vectors, shape [batch_size, embedding_dim]
        """
        # Trích xuất đặc trưng qua backbone ResNet18
        x = self.backbone(x)        # Shape: [batch_size, 512, 1, 1]
        x = x.view(x.size(0), -1)   # Flatten: [batch_size, 512]
        x = self.fc(x)              # Embedding: [batch_size, 128]
        return x

# Kiểm tra kiến trúc
cnn = CNNFeatureExtractor(embedding_dim=128)
print(">>> CNN Feature Extractor đã được xây dựng.")
print(f"    Output embedding: 128 chiều")

# Test thử với ảnh ngẫu nhiên
dummy_input = torch.randn(1, 3, 224, 224)  # 1 ảnh RGB 224x224
dummy_output = cnn(dummy_input)
print(f"    Test: Input {dummy_input.shape} -> Output {dummy_output.shape}")
print("\n>>> Bước 4 hoàn tất!")

---
## BƯỚC 5: XÂY DỰNG MẠNG SIAMESE (Photo Matching Network)

Mạng Siamese nhận vào **2 ảnh**, truyền qua **cùng một mạng CNN** (chia sẻ trọng số) để lấy 2 vector đặc trưng, sau đó tính **Euclidean Distance** giữa chúng.

```
Image A ──→ CNN ──→ Feature A ──┐
                 (shared)        ├──→ Euclidean Distance
Image B ──→ CNN ──→ Feature B ──┘
```

In [ ]:
class SiameseNetwork(nn.Module):
    """
    Mạng Siamese Network cho bài toán Photo Matching.
    
    Kiến trúc:
        1. Nhận vào 2 ảnh (Image A và Image B)
        2. Truyền cả 2 qua CÙNG MỘT mạng CNN (weight sharing)
        3. Nhận được 2 vector đặc trưng (Feature A và Feature B)
        4. Tính khoảng cách Euclidean giữa 2 vector
    
    Weight sharing là đặc trưng cốt lõi của Siamese Network:
    -> Đảm bảo 2 ảnh được xử lý bởi cùng một bộ lọc
    -> Cho phép so sánh công bằng giữa các đặc trưng
    """
    
    def __init__(self, embedding_dim=128):
        super(SiameseNetwork, self).__init__()
        
        # Chỉ tạo MỘT mạng CNN duy nhất (shared weights)
        self.cnn = CNNFeatureExtractor(embedding_dim=embedding_dim)
    
    def forward(self, img_a, img_b):
        """
        Forward pass cho cặp ảnh.
        
        Args:
            img_a (Tensor): Batch ảnh A, shape [batch_size, 3, 224, 224]
            img_b (Tensor): Batch ảnh B, shape [batch_size, 3, 224, 224]
        
        Returns:
            feat_a (Tensor): Vector đặc trưng ảnh A
            feat_b (Tensor): Vector đặc trưng ảnh B
            distance (Tensor): Khoảng cách Euclidean giữa 2 vector
        """
        # Trích xuất đặc trưng qua CÙNG MỘT mạng CNN
        feat_a = self.cnn(img_a)   # Feature vector cho ảnh A
        feat_b = self.cnn(img_b)   # Feature vector cho ảnh B
        
        # Tính khoảng cách Euclidean giữa 2 vector đặc trưng
        # ||feat_a - feat_b||_2
        distance = F.pairwise_distance(feat_a, feat_b, keepdim=True)
        
        return feat_a, feat_b, distance

# Khởi tạo mô hình và đưa lên device (CPU/GPU)
model = SiameseNetwork(embedding_dim=128).to(device)

print(">>> Siamese Network đã được xây dựng.")
print(f"    Device: {device}")

# Test thử
dummy_a = torch.randn(2, 3, 224, 224).to(device)
dummy_b = torch.randn(2, 3, 224, 224).to(device)
feat_a, feat_b, dist = model(dummy_a, dummy_b)
print(f"    Test: 2 ảnh -> Feature A {feat_a.shape}, Feature B {feat_b.shape}")
print(f"    Khoảng cách Euclidean: {dist.squeeze().tolist()}")
print("\n>>> Bước 5 hoàn tất!")

---
## BƯỚC 6: HÀM MẤT MÁT (Contrastive Loss) & OPTIMIZER

**Contrastive Loss** giúp mô hình học cách:
- **Kéo lại gần** các cặp ảnh giống nhau (label = 0) → minimize distance
- **Đẩy ra xa** các cặp ảnh khác nhau (label = 1) → maximize distance (tối thiểu = margin)

**Công thức:**
```
L = (1 - Y) × ½ × D² + Y × ½ × max(0, margin - D)²
```
- `Y = 0`: Cặp giống nhau → Loss = ½ × D² → Muốn D nhỏ
- `Y = 1`: Cặp khác nhau → Loss = ½ × max(0, margin - D)² → Muốn D ≥ margin

In [ ]:
class ContrastiveLoss(nn.Module):
    """
    Contrastive Loss cho Siamese Network.
    
    Công thức:
        L = (1 - Y) * 0.5 * D^2 + Y * 0.5 * max(0, margin - D)^2
    
    Trong đó:
        - D: Khoảng cách Euclidean giữa 2 feature vectors
        - Y: Label (0 = giống nhau, 1 = khác nhau)
        - margin: Ngưỡng khoảng cách tối thiểu cho cặp khác nhau
    """
    
    def __init__(self, margin=2.0):
        """
        Args:
            margin (float): Khoảng cách tối thiểu mong muốn giữa cặp ảnh khác nhau.
                           Giá trị càng lớn → mô hình càng cố đẩy cặp khác nhau ra xa.
        """
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
    
    def forward(self, distance, label):
        """
        Tính Contrastive Loss.
        
        Args:
            distance (Tensor): Khoảng cách Euclidean, shape [batch_size, 1]
            label (Tensor): Label (0=similar, 1=dissimilar), shape [batch_size]
        
        Returns:
            Tensor: Giá trị loss trung bình của batch
        """
        # Đảm bảo distance và label cùng shape
        distance = distance.squeeze()  # [batch_size]
        
        # Tính loss cho cặp giống nhau: (1 - Y) * 0.5 * D^2
        # Khi Y=0 (giống nhau): loss = 0.5 * D^2 → muốn D = 0
        similar_loss = (1 - label) * 0.5 * torch.pow(distance, 2)
        
        # Tính loss cho cặp khác nhau: Y * 0.5 * max(0, margin - D)^2
        # Khi Y=1 (khác nhau): loss = 0.5 * max(0, margin - D)^2 → muốn D >= margin
        dissimilar_loss = label * 0.5 * torch.pow(
            torch.clamp(self.margin - distance, min=0.0), 2
        )
        
        # Loss trung bình của batch
        loss = torch.mean(similar_loss + dissimilar_loss)
        
        return loss

# --- Khởi tạo Loss function và Optimizer ---
MARGIN = 2.0           # Ngưỡng margin cho Contrastive Loss
LEARNING_RATE = 0.0005  # Tốc độ học

criterion = ContrastiveLoss(margin=MARGIN)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(">>> Contrastive Loss và Optimizer đã được khởi tạo.")
print(f"    Margin: {MARGIN}")
print(f"    Optimizer: Adam (lr={LEARNING_RATE})")
print("\n>>> Bước 6 hoàn tất!")

---
## BƯỚC 7: HUẤN LUYỆN MÔ HÌNH (Training Loop)

Vòng lặp huấn luyện:
1. Lấy batch cặp ảnh từ DataLoader
2. Forward pass qua Siamese Network → tính distance
3. Tính Contrastive Loss
4. Backward pass → cập nhật trọng số
5. Lặp lại cho mỗi epoch

In [ ]:
# --- Cấu hình huấn luyện ---
NUM_EPOCHS = 20   # Số epoch huấn luyện

# Danh sách lưu loss để vẽ biểu đồ sau
train_losses = []

print(f"Bắt đầu huấn luyện: {NUM_EPOCHS} epochs")
print(f"{'='*60}")

# --- Vòng lặp huấn luyện ---
for epoch in range(NUM_EPOCHS):
    model.train()  # Chuyển model sang chế độ training
    running_loss = 0.0
    num_batches = 0
    
    # Duyệt qua từng batch trong train_loader
    for batch_idx, (img_a, img_b, label) in enumerate(tqdm(
        train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False
    )):
        # Đưa dữ liệu lên device (CPU/GPU)
        img_a = img_a.to(device)
        img_b = img_b.to(device)
        label = label.to(device)
        
        # --- Forward pass ---
        feat_a, feat_b, distance = model(img_a, img_b)
        
        # --- Tính loss ---
        loss = criterion(distance, label)
        
        # --- Backward pass ---
        optimizer.zero_grad()   # Xóa gradient cũ
        loss.backward()         # Tính gradient mới
        optimizer.step()        # Cập nhật trọng số
        
        running_loss += loss.item()
        num_batches += 1
    
    # Tính loss trung bình của epoch
    epoch_loss = running_loss / max(num_batches, 1)
    train_losses.append(epoch_loss)
    
    # In kết quả mỗi epoch
    print(f"Epoch [{epoch+1:3d}/{NUM_EPOCHS}] | Loss: {epoch_loss:.4f}")

print(f"{'='*60}")
print(">>> Huấn luyện hoàn tất!")

# --- Lưu model đã train ---
model_path = os.path.join(MODEL_DIR, "siamese_model.pth")
torch.save(model.state_dict(), model_path)
print(f">>> Model đã được lưu tại: {model_path}")

---
## BƯỚC 8: ĐÁNH GIÁ KẾT QUẢ

- Tính **Accuracy** trên tập test
- Vẽ **biểu đồ Loss** qua các epoch

In [ ]:
# --- 8.1: Vẽ biểu đồ Loss ---

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', linewidth=2, markersize=5)
plt.title('Training Loss qua các Epoch', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Contrastive Loss', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(train_losses) + 1))
plt.tight_layout()
plt.show()

print(f"Loss ban đầu:  {train_losses[0]:.4f}")
print(f"Loss cuối cùng: {train_losses[-1]:.4f}")
print(f"Giảm: {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%")

In [ ]:
# --- 8.2: Tính Accuracy trên tập Test ---

THRESHOLD = 1.0  # Ngưỡng phân loại: distance < threshold → giống nhau

model.eval()  # Chuyển model sang chế độ đánh giá

correct = 0
total = 0

# Không tính gradient khi đánh giá (tiết kiệm bộ nhớ)
with torch.no_grad():
    for img_a, img_b, label in test_loader:
        img_a = img_a.to(device)
        img_b = img_b.to(device)
        label = label.to(device)
        
        # Forward pass
        _, _, distance = model(img_a, img_b)
        distance = distance.squeeze()
        
        # Dự đoán: distance < threshold → giống nhau (predicted=0)
        #           distance >= threshold → khác nhau (predicted=1)
        predicted = (distance >= THRESHOLD).float()
        
        # Đếm số dự đoán đúng
        correct += (predicted == label).sum().item()
        total += label.size(0)

accuracy = correct / max(total, 1) * 100
print(f"{'='*40}")
print(f"KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST")
print(f"{'='*40}")
print(f"Threshold: {THRESHOLD}")
print(f"Tổng số cặp test: {total}")
print(f"Dự đoán đúng: {correct}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"{'='*40}")
print("\n>>> Bước 8 hoàn tất!")

---
## BƯỚC 9: DEMO - DỰ ĐOÁN VỚI 2 ẢNH BẤT KỲ

Hàm nhận vào 2 đường dẫn ảnh bất kỳ, đưa qua mô hình đã train và in ra kết quả:
- Hai ảnh có giống nhau không?
- Khoảng cách Euclidean giữa 2 ảnh

In [ ]:
def predict(img1_path, img2_path, model, transform, threshold=1.0, device='cpu'):
    """
    Dự đoán 2 ảnh có giống nhau không bằng Siamese Network.
    
    Args:
        img1_path (str): Đường dẫn đến ảnh thứ 1
        img2_path (str): Đường dẫn đến ảnh thứ 2
        model: Mô hình Siamese Network đã train
        transform: Các phép biến đổi ảnh
        threshold (float): Ngưỡng phân loại (distance < threshold → giống nhau)
        device: Thiết bị tính toán (CPU/GPU)
    
    Returns:
        dict: Kết quả dự đoán gồm distance, is_similar, label
    """
    model.eval()  # Chế độ đánh giá
    
    # --- Đọc và tiền xử lý 2 ảnh ---
    img1 = Image.open(img1_path).convert('RGB')
    img2 = Image.open(img2_path).convert('RGB')
    
    # Áp dụng transforms và thêm batch dimension
    img1_tensor = transform(img1).unsqueeze(0).to(device)  # [1, 3, 224, 224]
    img2_tensor = transform(img2).unsqueeze(0).to(device)  # [1, 3, 224, 224]
    
    # --- Forward pass ---
    with torch.no_grad():
        feat1, feat2, distance = model(img1_tensor, img2_tensor)
    
    distance_value = distance.item()
    is_similar = distance_value < threshold
    
    # --- Hiển thị kết quả ---
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    # Hiển thị ảnh 1
    axes[0].imshow(img1)
    axes[0].set_title('Ảnh 1', fontsize=12)
    axes[0].axis('off')
    
    # Hiển thị ảnh 2
    axes[1].imshow(img2)
    axes[1].set_title('Ảnh 2', fontsize=12)
    axes[1].axis('off')
    
    # Tiêu đề kết quả
    result_text = "GIỐNG NHAU ✓" if is_similar else "KHÁC NHAU ✗"
    result_color = "green" if is_similar else "red"
    
    fig.suptitle(
        f"Kết quả: {result_text}\n"
        f"Khoảng cách Euclidean: {distance_value:.4f} (Ngưỡng: {threshold})",
        fontsize=14, fontweight='bold', color=result_color
    )
    
    plt.tight_layout()
    plt.show()
    
    # In kết quả ra console
    print(f"{'='*50}")
    print(f"KẾT QUẢ DỰ ĐOÁN")
    print(f"{'='*50}")
    print(f"Ảnh 1: {img1_path}")
    print(f"Ảnh 2: {img2_path}")
    print(f"Khoảng cách Euclidean: {distance_value:.4f}")
    print(f"Ngưỡng (threshold):    {threshold}")
    print(f"Kết quả: {result_text}")
    print(f"{'='*50}")
    
    return {
        'distance': distance_value,
        'is_similar': is_similar,
        'label': result_text
    }

print(">>> Hàm predict() đã sẵn sàng.")
print(">>> Bước 9 hoàn tất!")

In [ ]:
# --- DEMO: Thử dự đoán với 2 ảnh ---
# Thay đường dẫn bên dưới bằng ảnh thật của bạn

# Ví dụ: So sánh 2 ảnh giống nhau
# result = predict(
#     img1_path="data/input/similar/pair_001_a.jpg",
#     img2_path="data/input/similar/pair_001_b.jpg",
#     model=model,
#     transform=transform,
#     threshold=1.0,
#     device=device
# )

# Ví dụ: So sánh 2 ảnh khác nhau
# result = predict(
#     img1_path="data/input/dissimilar/pair_001_a.jpg",
#     img2_path="data/input/dissimilar/pair_001_b.jpg",
#     model=model,
#     transform=transform,
#     threshold=1.0,
#     device=device
# )

print(">>> Bỏ comment các dòng trên và thay đường dẫn ảnh để chạy demo!")

In [ ]:
# --- Load model đã lưu (nếu muốn dùng lại sau) ---

# model_loaded = SiameseNetwork(embedding_dim=128).to(device)
# model_loaded.load_state_dict(torch.load("models/siamese_model.pth", map_location=device))
# model_loaded.eval()
# print(">>> Model đã được load thành công!")

# Sau đó dùng predict() với model_loaded:
# result = predict("path/to/img1.jpg", "path/to/img2.jpg", model_loaded, transform, 1.0, device)